In [7]:
import sys
sys.path.insert(0, "../")

import pandas as pd
import src.openmeteo as api
import src.queimada as q
import requests
import time
from datetime import datetime, timedelta

In [8]:
df = pd.read_csv('../bdqueimadas.csv')

In [9]:
# Colunas não utilizadas pra treinamento
df = df.drop(columns=['Satelite', 'Pais', 'Estado', 'FRP', 'Latitude', 'Longitude'])

# Apaga os dados dos biomas que não sejam mata atlântica e cerrado
df = df[df["Bioma"].isin(["Mata Atlântica", "Cerrado"])]

# Converte o bioma em um valor numérico baseado no tipo de vegetação.
dados = {'Mata Atlântica': 1.7, 'Cerrado': 3}
df['Bioma'] = df['Bioma'].map(dados)

# Filtra os dados quebrados
df = df[df["DiaSemChuva"] >= 0]
df = df[df["RiscoFogo"] >= 0]

# Garante que a coluna é datetime
df['DataHora'] = pd.to_datetime(df['DataHora'])

# Cria uma coluna temporária com o dia para filtrar as repetidas do mesmo dia (mantendo sua lógica original)
df['Data_Temp'] = df['DataHora'].dt.date
df = df.drop_duplicates(subset=['Data_Temp', 'Municipio'], keep='first')
df = df.drop(columns=['Data_Temp']) # Remove a coluna temporária

# Agora, converte a DataHora para o período mensal (Ex: 2025-09)
df['DataHora'] = df['DataHora'].dt.to_period('M')

# Agrupa por Mês e Município, e tira a média de todas as outras colunas (DiaSemChuva, RiscoFogo, Bioma)
df = df.groupby(['DataHora', 'Municipio']).mean().reset_index()

# Ordena por lugar e data
df = df.sort_values(by=['Municipio', 'DataHora'])
df = df.reset_index(drop=True)

# !APENAS PARA TESTES
df = df.sample(frac=0.05)

In [10]:
def meses_faltantes(municipio):
    existentes = df_agrupado[municipio] if municipio in df_agrupado else []
    return [mes for mes in todos_os_meses if mes not in existentes]

def obter_com_retry(dia, municipio, max_tentativas=10):
    """Devolve o resultado da API ou None se esgotar as tentativas."""
    for tentativa in range(max_tentativas):
        try:
            return api.obter_dados_clima(dia, municipio)
        except requests.exceptions.RequestException:
            if tentativa == max_tentativas - 1:
                return None
            espera = 2 ** (tentativa + 1)
            print(f'Erro: tentando de novo em {espera} segundos')
            time.sleep(espera)

def mostrar_progresso(municipio, mes, status):
    decorrido = time.perf_counter() - inicio
    restante = (decorrido / processados) * (total - processados)
    print(
        f'[{processados / total * 100:5.1f}%] {processados}/{total} | '
        f'decorrido: {timedelta(seconds=int(decorrido))} | '
        f'faltam ~{timedelta(seconds=int(restante))} | '
        f'{status}: {municipio} {mes}'
    )

In [11]:
# Adiciona a coluna "fogo" = 1 para todos os dados do bdqueimadas (dias do ano em que cada local REGISTROU foco de calor)
df['Fogo'] = 1

# Adiciona as colunas faltantes, onde "fogo" = 0 (dias do ano em que cada local NÃO registrou foco de calor)

municipios = df['Municipio'].unique()
todos_os_meses = (
    pd.date_range(start='2025-6', end='2025-12', freq='MS')
    .strftime('%Y-%m')
    .tolist()
)
df_agrupado = df.groupby('Municipio')['DataHora'].unique()

if 'PeriodoSecura' not in df.columns:
    df['PeriodoSecura'] = pd.NA   # entra no final do df

# Assumindo que mes e municipio são as duas primeiras colunas (mesma ordem de novas_linhas)
col_mes, col_mun = df.columns[0], df.columns[1]

vegetacao_por_municipio = df.groupby('Municipio')['Bioma'].mean().to_dict()
total = sum(len(meses_faltantes(m)) for m in municipios)
processados = 0
novas_linhas = []
pulados = []
inicio = time.perf_counter()

try:
    # 1) Linhas que já estavam no df: recalcula o PSE com a função
    for idx in df.index[df['PeriodoSecura'].isna()]:
        mes = str(mes)
        municipio = df.at[idx, col_mun]
        dia = datetime.strptime(mes, '%Y-%m').date()

        resultado = obter_com_retry(dia, municipio)
        processados += 1

        if resultado is None:
            pulados.append((municipio, mes))
            mostrar_progresso(municipio, mes, 'PULADO')
            continue

        precs = {k: v for k, v in resultado.items() if k.startswith('prec')}
        df.at[idx, 'PeriodoSecura'] = q.periodo_de_secura(precs)
        mostrar_progresso(municipio, mes, 'ok')

    # 2) Meses novos
    for municipio in municipios:
        for mes in meses_faltantes(municipio):
            dia = datetime.strptime(mes, '%Y-%m').date()

            resultado = obter_com_retry(dia, municipio)
            processados += 1

            if resultado is None:
                pulados.append((municipio, mes))
                mostrar_progresso(municipio, mes, 'PULADO')
                continue

            vegetacao = vegetacao_por_municipio[municipio]
            umidade = resultado['umidade']
            temperatura = resultado['temperatura']
            precipitacao = resultado['precipitacao']
            precs = {k: v for k, v in resultado.items() if k.startswith('prec')}

            pse = q.periodo_de_secura(precs)
            rf = q.risco_fogo(vegetacao, pse, umidade, temperatura)

            # Ordem = colunas do df, com PeriodoSecura por último
            novas_linhas.append([
                mes, municipio, vegetacao,
                resultado['diasemchuva'],   # DiaSemChuva
                precipitacao, rf, 0,
                pse,                        # PeriodoSecura
            ])
            mostrar_progresso(municipio, mes, 'ok')

finally:
    # Mesmo com Ctrl+C ou erro inesperado, o que já foi coletado entra no df
    if novas_linhas:
        df = pd.concat(
            [df, pd.DataFrame(novas_linhas, columns=df.columns)],
            ignore_index=True,
        )


[  0.1%] 1/1033 | decorrido: 0:00:01 | faltam ~0:32:51 | ok: SANTA CRUZ DA CONCEIÇÃO 2025-09
[  0.2%] 2/1033 | decorrido: 0:00:03 | faltam ~0:33:12 | ok: COSMORAMA 2025-09
[  0.3%] 3/1033 | decorrido: 0:00:05 | faltam ~0:33:11 | ok: PEREIRA BARRETO 2025-09
[  0.4%] 4/1033 | decorrido: 0:00:07 | faltam ~0:33:11 | ok: DUARTINA 2025-09
[  0.5%] 5/1033 | decorrido: 0:00:09 | faltam ~0:33:13 | ok: BOITUVA 2025-09
[  0.6%] 6/1033 | decorrido: 0:00:11 | faltam ~0:33:12 | ok: RIO DAS PEDRAS 2025-09
[  0.7%] 7/1033 | decorrido: 0:00:13 | faltam ~0:33:09 | ok: ITABERÁ 2025-09
[  0.8%] 8/1033 | decorrido: 0:00:15 | faltam ~0:33:04 | ok: TRABIJU 2025-09
[  0.9%] 9/1033 | decorrido: 0:00:17 | faltam ~0:33:02 | ok: SÃO JOSÉ DO RIO PARDO 2025-09
[  1.0%] 10/1033 | decorrido: 0:00:19 | faltam ~0:33:01 | ok: RIBEIRÃO PIRES 2025-09
[  1.1%] 11/1033 | decorrido: 0:00:21 | faltam ~0:32:55 | ok: SANTA CRUZ DO RIO PARDO 2025-09
[  1.2%] 12/1033 | decorrido: 0:00:23 | faltam ~0:33:04 | ok: BOCAINA 2025-09
[ 

In [12]:
# Número total de municípios únicos
total_municipios = df['Municipio'].nunique()

# Conta quantas vezes cada município aparece e depois tira a média
media_aparicoes = df['Municipio'].value_counts().mean()

print(f'Número de municípios: {total_municipios}')
print(f'Média de focos por município: {media_aparicoes:.2f}')

# Salva o dataframe em um arquivo CSV
df.to_csv("../out.csv", index=False)

Número de municípios: 167
Média de focos por município: 7.31
